In [1]:
import os
import tensorflow as tf
import tensorflow_model_analysis as tfma
import tfx.v1 as tfx

# Import komponen standar TFX
from tfx.components import (
    CsvExampleGen, StatisticsGen, SchemaGen, ExampleValidator,
    Transform, Trainer, Evaluator, Pusher
)

# Import khusus untuk Resolver
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.experimental import latest_blessed_model_resolver

# Import Channel dan Artifacts
from tfx.types import Channel 
from tfx.types.standard_artifacts import Model, ModelBlessing

# Import Proto, BeamDagRunner, dan Metadata
from tfx.proto import trainer_pb2, pusher_pb2
from tfx.orchestration.beam.beam_dag_runner import BeamDagRunner
from tfx.orchestration import metadata  # <--- TAMBAHKAN BARIS INI

print(f"TensorFlow Version: {tf.__version__}")
print(f"TFX Version: {tfx.__version__}")
print("✅ Import berhasil!")

C:\Users\Dragon\miniconda3\envs\mlops-tfx\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
C:\Users\Dragon\miniconda3\envs\mlops-tfx\lib\site-packages\apache_beam\runners\portability\stager.py:63: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


TensorFlow Version: 2.10.1
TFX Version: 1.11.0
✅ Import berhasil!


In [2]:
# Ganti dengan username Dicoding Anda
USERNAME = 'andyid'  

# Konfigurasi pipeline
PIPELINE_NAME = f"{USERNAME}-pipeline"
PIPELINE_ROOT = os.path.join('.', PIPELINE_NAME)
METADATA_PATH = os.path.join(PIPELINE_ROOT, 'metadata', 'metadata.db')

# Path dataset (folder berisi file CSV Anda)
DATA_ROOT = 'data'

# Path untuk model yang akan di-serve
SERVING_MODEL_DIR = 'serving_model'

print(f"Pipeline Name: {PIPELINE_NAME}")
print(f"Data Root: {DATA_ROOT}")

Pipeline Name: andyid-pipeline
Data Root: data


In [3]:
# 1. ExampleGen
example_gen = CsvExampleGen(input_base=DATA_ROOT)

# 2. StatisticsGen
statistics_gen = StatisticsGen(examples=example_gen.outputs['examples'])

# 3. SchemaGen
schema_gen = SchemaGen(
    statistics=statistics_gen.outputs['statistics'],
    infer_feature_shape=True
)

# 4. ExampleValidator
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema']
)

# 5. Transform
transform = Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file='modules/transform.py'
)

# 6. Trainer
trainer = Trainer(
    module_file='modules/trainer.py',
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(num_steps=2000),
    eval_args=trainer_pb2.EvalArgs(num_steps=500)
)

# 7. Resolver - PERBAIKAN DI SINI
model_resolver = Resolver(
    strategy_class=latest_blessed_model_resolver.LatestBlessedModelResolver,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
).with_id('latest_blessed_model_resolver')

# 8. Evaluator
evaluator = Evaluator(
    examples=example_gen.outputs['examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=tfma.EvalConfig(
        model_specs=[tfma.ModelSpec(label_key='target')],
        metrics_specs=[
            tfma.MetricsSpec(metrics=[
                tfma.MetricConfig(class_name='ExampleCount'),
                tfma.MetricConfig(class_name='BinaryAccuracy'),
                tfma.MetricConfig(class_name='AUC'),
            ])
        ],
        slicing_specs=[tfma.SlicingSpec()]
    )
)

# 9. Pusher
pusher = Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    )
)

print("✅ Semua 9 komponen pipeline berhasil didefinisikan!")

✅ Semua 9 komponen pipeline berhasil didefinisikan!


In [ ]:
# Kumpulkan semua komponen
components = [
    example_gen,
    statistics_gen,
    schema_gen,
    example_validator,
    transform,
    trainer,
    model_resolver,
    evaluator,
    pusher
]

# Buat pipeline configuration
pipeline = tfx.dsl.Pipeline(
    pipeline_name=PIPELINE_NAME,
    pipeline_root=PIPELINE_ROOT,
    components=components,
    metadata_connection_config=metadata.sqlite_metadata_connection_config(METADATA_PATH),
    enable_cache=True,
)

# Jalankan pipeline menggunakan Apache Beam
BeamDagRunner().run(pipeline)

print(f"\n✅ Pipeline '{PIPELINE_NAME}' berhasil dijalankan dengan Apache Beam!")
print(f"📁 Artifact tersimpan di: {PIPELINE_ROOT}")

In [ ]:
# Tampilkan hasil evaluasi model
import ipywidgets as widgets

model_run_dir = os.path.join(PIPELINE_ROOT, 'Evaluator', 'output')
if os.path.exists(model_run_dir):
    tfma_result = tfma.load_eval_result(model_run_dir)
    tfma.view.render_slicing_metrics(tfma_result)
    print("✅ Visualisasi metrik evaluasi berhasil dimuat!")
else:
    print("⚠️ Hasil evaluasi belum tersedia. Pastikan pipeline sudah selesai dijalankan.")